In [ ]:
# Run this cell first to install required packages in JupyterLite
%pip install -q numpy matplotlib ipywidgets scikit-learn


# Figure 18.11: Reparameterization trick for gradient-based training
A **Variational Autoencoder** (VAE) encodes each input as a distribution, not a point:
$$q_\theta(z|x) = \mathcal{N}(\mu, \sigma^2)$$
A sample is drawn using the **reparameterization trick**:
$$z = \mu + \sigma \cdot \varepsilon, \quad \varepsilon \sim \mathcal{N}(0,1)$$

The training objective is the **ELBO**:
$$\mathcal{L}_{\text{ELBO}} = \underbrace{\mathbb{E}[\log p_\phi(x|z)]}_{\text{reconstruction}} - \beta \underbrace{D_{KL}[q_\theta(z|x) \| p(z)]}_{\text{KL regularization}}$$
$$D_{KL}[\mathcal{N}(\mu,\sigma^2) \| \mathcal{N}(0,1)] = \tfrac{1}{2}(\mu^2 + \sigma^2 - \log\sigma^2 - 1)$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

def normal_pdf(x, mu, sig):
    return np.exp(-0.5*((x-mu)/sig)**2) / (sig*np.sqrt(2*np.pi))

def kl_div(mu, sig):
    """KL[N(mu,sig^2) || N(0,1)]"""
    return 0.5*(mu**2 + sig**2 - 2*np.log(sig) - 1)


def draw_vae(mu=1.5, sig=1.0, beta=1.0, rec_loss=0.5):
    z = np.linspace(-6, 6, 400)
    prior = normal_pdf(z, 0, 1)
    post  = normal_pdf(z, mu, sig)
    kl    = kl_div(mu, sig)
    elbo  = -(rec_loss + beta*kl)

    rng   = np.random.default_rng(42)
    eps   = rng.normal(0, 1, 300)
    z_samp= mu + sig*eps

    mu_range = np.linspace(-3, 3, 200)
    kl_curve  = kl_div(mu_range, sig)
    elbo_curve= -(rec_loss + beta*kl_curve)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # 1. Gaussians
    axes[0].fill_between(z, prior, alpha=0.15, color='#2563eb')
    axes[0].fill_between(z, post,  alpha=0.15, color='#dc2626')
    axes[0].plot(z, prior, '#2563eb', lw=2, label='Prior N(0,1)')
    axes[0].plot(z, post,  '#dc2626', lw=2.5, label=f'Posterior N({mu:.1f},{sig:.2f}²)')
    axes[0].axvline(mu, color='#dc2626', ls=':', lw=2)
    axes[0].set_xlabel('z'); axes[0].set_ylabel('Density')
    axes[0].set_title('Posterior vs Prior', fontsize=11)
    axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

    # 2. Reparameterization
    eps_grid = np.linspace(-3, 3, 100)
    axes[1].scatter(eps[:100], z_samp[:100], s=15, color='#dc2626', alpha=0.5, label='Samples')
    axes[1].plot(eps_grid, mu + sig*eps_grid, '#7c3aed', lw=2, label=f'z=μ+σε  (μ={mu:.1f},σ={sig:.2f})')
    axes[1].axhline(0, color='#94a3b8', lw=1); axes[1].axvline(0, color='#94a3b8', lw=1)
    axes[1].set_xlabel('ε ~ N(0,1)'); axes[1].set_ylabel('z = μ + σ·ε')
    axes[1].set_title('Reparameterization Trick', fontsize=11)
    axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

    # 3. ELBO vs mu
    axes[2].plot(mu_range, beta*kl_curve, '#dc2626', lw=2, label=f'β·KL (β={beta:.1f})')
    axes[2].plot(mu_range, elbo_curve, '#059669', lw=2.5, label='ELBO = −Rec − β·KL')
    axes[2].axhline(-rec_loss, color='#7c3aed', ls='--', lw=1.5, label=f'−Rec (={-rec_loss:.2f})')
    axes[2].axvline(mu, color='#d97706', ls=':', lw=2, label=f'current μ={mu:.1f}')
    axes[2].set_xlabel('μ'); axes[2].set_ylabel('Value')
    axes[2].set_title('ELBO Components vs μ', fontsize=11)
    axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)

    plt.suptitle(f'KL={kl:.4f}  ELBO={elbo:.4f}  β={beta:.1f}', fontsize=10, y=0.0)
    plt.tight_layout(); plt.show()
    print(f'KL[N({mu:.2f},{sig:.2f}²)||N(0,1)] = {kl:.4f}')
    print(f'ELBO = −{rec_loss:.3f} − {beta:.1f}×{kl:.4f} = {elbo:.4f}')


widgets.interact(draw_vae,
    mu=widgets.FloatSlider(min=-3, max=3, step=0.1, value=1.5, description='μ', continuous_update=False),
    sig=widgets.FloatSlider(min=0.1, max=3, step=0.05, value=1.0, description='σ', continuous_update=False),
    beta=widgets.FloatSlider(min=0, max=3, step=0.1, value=1.0, description='β (KL weight)', continuous_update=False),
    rec_loss=widgets.FloatSlider(min=0.01, max=2, step=0.01, value=0.5, description='Rec loss', continuous_update=False),
)